# MLPKriging — Deep Kernel Learning (R)

**MLPKriging** combines a multi-layer perceptron (MLP) feature map with Gaussian-process
regression. The MLP takes *all* input dimensions jointly and maps them into a learned
`d_out`-dimensional feature space, after which a standard Kriging model is fitted.
This is also known as *Deep Kernel Learning*.

Steps:
1. Install rlibkriging (run once)
2. Load rlibkriging
3. Define the Branin function and plot it
4. Build a space-filling design and evaluate it
5. Fit an `MLPKriging` model
6. Predict on a fine grid and plot mean + uncertainty
7. Inspect model parameters

## 0. Installation

```r
install.packages('rlibkriging')   # from CRAN or local build
```

## 1. Load rlibkriging

In [ ]:
library(rlibkriging)

## 2. Branin function

The Branin function is a standard benchmark for surrogate modelling, defined on $[0,1]^2$
(rescaled from its canonical domain $[-5, 10] \times [0, 15]$).

In [ ]:
branin <- function(X) {
  X  <- as.matrix(X)
  x1 <- X[, 1] * 15 - 5
  x2 <- X[, 2] * 15
  (x2 - 5 / (4 * pi^2) * x1^2 + 5 / pi * x1 - 6)^2 +
    10 * (1 - 1 / (8 * pi)) * cos(x1) + 10
}

# 50x50 evaluation grid
grid_x <- seq(0, 1, length.out = 50)
grid   <- as.matrix(expand.grid(x1 = grid_x, x2 = grid_x))
z_true <- matrix(branin(grid), 50, 50)

image(grid_x, grid_x, z_true, col = terrain.colors(30),
      main = "True Branin function",
      xlab = expression(x[1]), ylab = expression(x[2]))
contour(grid_x, grid_x, z_true, add = TRUE, col = "grey30", nlevels = 10)

## 3. Design of experiments

We sample $n = 30$ points using a random Latin Hypercube Design.

In [ ]:
set.seed(42)

lhs <- function(n, d) {
  X <- matrix(0, n, d)
  for (j in 1:d)
    X[, j] <- (sample.int(n) - 1 + runif(n)) / n
  X
}

n <- 30
X <- lhs(n, 2)
y <- branin(X)

image(grid_x, grid_x, z_true, col = terrain.colors(30),
      main = paste(n, "LHS design points on Branin"),
      xlab = expression(x[1]), ylab = expression(x[2]))
contour(grid_x, grid_x, z_true, add = TRUE, col = "grey30", nlevels = 10)
points(X[, 1], X[, 2], pch = 21, bg = "white", cex = 1.2)

## 4. Fit an MLPKriging model

We use `hidden_dims = c(16, 8)` (two hidden layers), `d_out = 2` (2D feature space),
and `activation = "selu"`.

In [ ]:
mk <- MLPKriging(
  y, X,
  hidden_dims = c(16, 8),
  d_out = 2,
  activation = "selu",
  kernel = "matern5_2",
  optim = "BFGS+Adam"
)
print(mk)

## 5. Predict on a fine grid

In [ ]:
p <- predict(mk, grid, return_stdev = TRUE)
z_mean <- matrix(p$mean,  50, 50)
z_sd   <- matrix(p$stdev, 50, 50)

brks <- seq(min(z_true, z_mean), max(z_true, z_mean), length.out = 30)
cols <- terrain.colors(length(brks) - 1)

par(mfrow = c(1, 2))

image(grid_x, grid_x, z_true, breaks = brks, col = cols,
      main = "True Branin", xlab = expression(x[1]), ylab = expression(x[2]))
contour(grid_x, grid_x, z_true, add = TRUE, col = "grey30", nlevels = 10)
points(X[, 1], X[, 2], pch = 21, bg = "white")

image(grid_x, grid_x, z_mean, breaks = brks, col = cols,
      main = "MLPKriging mean", xlab = expression(x[1]), ylab = expression(x[2]))
contour(grid_x, grid_x, z_mean, add = TRUE, col = "grey30", nlevels = 10)
points(X[, 1], X[, 2], pch = 21, bg = "white")

par(mfrow = c(1, 1))

In [ ]:
# Posterior standard deviation (uncertainty)
filled.contour(
  grid_x, grid_x, z_sd,
  color.palette = function(n) hcl.colors(n, "YlOrRd", rev = TRUE),
  main = "MLPKriging std dev (uncertainty)",
  xlab = expression(x[1]), ylab = expression(x[2]),
  plot.axes = {
    axis(1); axis(2)
    points(X[, 1], X[, 2], pch = 21, bg = "white", cex = 1.2)
  }
)

## 6. Model inspection

Key fitted parameters: length-scales $\theta$, variance $\sigma^2$, log-likelihood,
hidden layer architecture, and activation function.

In [ ]:
cat("Kernel       :", kernel(mk), "\n")
cat("Theta (range):", round(theta(mk), 4), "\n")
cat("Sigma2       :", round(sigma2(mk), 4), "\n")
cat("LogLikelihood:", round(logLikelihood(mk), 4), "\n")
cat("Feature dim  :", feature_dim(mk), "\n")
cat("Hidden dims  :", hidden_dims(mk), "\n")
cat("Activation   :", activation(mk), "\n")